# Automated profiling: a full EDA report in one line

_Data Wrangling_

**Point a profiler at a fresh dataset and get types, missingness, correlations, and alerts in seconds — then dig in by hand.**

You already know the manual inspection battery: load the frame, then call .info(),
       .describe(), .nunique(), .isna().sum() and build a mental model
       of every column. A profiler automates that whole battery and adds the parts that are tedious to
       do by hand &mdash; full distributions, a correlation matrix, duplicate-row counts &mdash; then it
       flags the problems for you.

       What you get back, per column and for the frame as a whole:

---

This notebook is a practice scaffold for the **Automated profiling: a full EDA report in one line** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q ydata-profiling
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — ydata-profiling + missingno + sweetviz

In [ ]:
import seaborn as sns
from ydata_profiling import ProfileReport   # formerly: pandas-profiling
import missingno as msno
import sweetviz as sv

# --- A real-ish frame with mixed dtypes and genuine missingness ---
df = sns.load_dataset('titanic')             # 891 rows; 'age' and 'deck' are missing-heavy

# === ydata-profiling: the whole inspection battery + alerts in ONE call ===
# Full report (per-column stats/distributions, missingness, duplicates,
# correlations, interactions, and an ALERTS section at the top):
ProfileReport(df, title='Titanic profile').to_file('report.html')

# On a BIG dataset, sample first and/or pass minimal=True to skip the
# expensive correlation/interaction sections:
ProfileReport(df.sample(500, random_state=0), minimal=True).to_file('report_fast.html')

# Read the alerts programmatically instead of opening the HTML:
report = ProfileReport(df, minimal=True)
for alert in report.get_description().alerts:
    print(alert)                             # e.g. [HIGH_CARDINALITY] deck, [MISSING] age ...

# === missingno: SEE the missingness pattern (the focused tool) ===
msno.matrix(df)        # one strip per column; gaps mark missing values
msno.heatmap(df)       # do two columns tend to be missing on the SAME rows?

# === sweetviz: COMPARE two frames side by side (its specialty) ===
survived    = df[df['survived'] == 1]
not_survived = df[df['survived'] == 0]
sv.compare([survived, 'Survived'], [not_survived, 'Did not survive']).show_html('compare.html')

# A profiler is the FAST FIRST PASS: it tells you WHERE to look.
# You then do the question-driven EDA (targeted plots/groupbys) by hand.

## Visualize the data & results

_Point a ydata-profiling-style profiler at a deliberately messy real frame — what KINDS of alerts does it raise, and how many of each?_

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Real bundled dataset: 569 cell-nucleus measurements.
df = load_breast_cancer(as_frame=True).frame.copy()
n = len(df)
rng = np.random.RandomState(0)

# --- Rough it up to look like a raw export ---
df['const_col'] = 1.0                       # a column with ONE value
df['record_id'] = np.arange(n)              # a unique ID column
for col, frac in [('mean radius', 0.30), ('mean texture', 0.12), ('mean area', 0.05)]:
    idx = rng.choice(n, int(frac * n), replace=False)
    df.loc[idx, col] = np.nan                # inject missingness

# --- Apply ydata-profiling-style alert RULES and count each kind ---
alerts = {'High correlation': 0, 'Missing': 0, 'Unique (ID)': 0,
          'Constant': 0, 'Imbalance': 0}

corr = df.select_dtypes('number').corr().abs()
high = set()
for i, a in enumerate(corr.columns):
    for b in corr.columns[i + 1:]:
        if corr.loc[a, b] > 0.9:            # near-duplicate columns
            high.add(a); high.add(b)
alerts['High correlation'] = len(high)

for c in df.columns:
    nu = df[c].nunique(dropna=True)
    if nu == 1:            alerts['Constant']   += 1     # one distinct value
    if nu == n:            alerts['Unique (ID)'] += 1    # distinct == n_rows
    if df[c].isna().any(): alerts['Missing']    += 1     # any nulls
    if 0 < nu <= 20 and df[c].value_counts(normalize=True).iloc[0] > 0.8:
        alerts['Imbalance'] += 1                          # one category dominates

print(alerts)
# -> {'High correlation': 14, 'Missing': 3, 'Unique (ID)': 1,
#     'Constant': 1, 'Imbalance': 1}

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your ProfileReport(df) call has been running for fifteen minutes on a 20-million-row table and the kernel is thrashing. What is happening, and what are the two standard fixes?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that a full report computes the pairwise correlation matrix and a per-column histogram for every column. — _Pairwise correlations grow with the number of column pairs and scan all rows, so on tens of millions of rows the work explodes in time and memory._
- Sample the rows before profiling: ProfileReport(df.sample(50_000, random_state=0)). — _A profiler is a first-pass smoke test; 50k rows show the same types, missingness, and broad correlations as 20M without the cost._
- Or pass minimal=True to skip the expensive sections. — _minimal=True drops correlations/interactions and keeps the cheap per-column summaries, which is often all you need for first contact._

**Answer:** The correlation/interaction and per-column histogram work is quadratic-ish in columns
        and scans every row, so an un-sampled 20M-row frame is slow or crashes. The two fixes:
        sample first (df.sample(50_000)) and/or pass minimal=True to
        skip the expensive correlation section. Never profile a huge dataset un-sampled.

</details>

**Problem 2.** A profiler raises a "High correlation" alert between monthly_charge and total_billed in a churn dataset. A teammate wants to drop one column immediately. What should you check first, and what are the three things this alert can mean?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Remember that "high correlation" only means the two columns move together — it is not a verdict. — _The alert is a pointer to look, not a conclusion; correlation is not causation and not always redundancy._
- Inspect the pair: are they near-duplicates, is one derived from the target, or is it coincidence? — _total_billed may be monthly_charge times tenure (redundant), or it may encode information about churn timing (a leak), or be unrelated chance._
- Decide based on what you find: drop a true duplicate, remove a leaky feature, or keep both if it is coincidental and both are legitimately predictive. — _Acting on the alert blindly can either throw away signal or leave a leak in place._

**Answer:** First look at the pair &mdash; the alert is a prompt, not a decision. A high-correlation
        alert can mean (1) a redundant duplicate column (drop one), (2) a leak, where one feature
        secretly encodes the target or its timing (remove it), or (3) coincidence between two genuinely
        useful features (keep both). Correlation is not causation; investigate before dropping.

</details>

**Problem 3.** You profile a customer table and the report looks great, so you email the HTML file to a partner team. Why might this be a serious mistake, and how should you have prepared the frame first?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that the report embeds sample rows and the most-frequent values for each column. — _Those samples and top-value lists can contain names, emails, phone numbers, or account IDs verbatim._
- Recognize that the HTML is therefore a copy of sensitive raw data, now in an emailable file. — _PII (Personally Identifiable Information) leaving its controlled system is a privacy and compliance violation._
- Drop or mask sensitive columns before profiling: ProfileReport(df.drop(columns=pii_cols)). — _Profiling only the non-sensitive columns produces a shareable report with no personal data embedded._

**Answer:** The report embeds raw sample rows and top values, so a shared HTML can leak PII
        (Personally Identifiable Information) &mdash; names, emails, IDs &mdash; into an emailable file.
        Before profiling, drop or mask the sensitive columns and treat the report with the same care as
        the underlying data.

</details>